# ICU Preprocessing Pipeline with ReciPies

This notebook demonstrates a **complete end-to-end preprocessing pipeline** for ICU time-series data using ReciPies (see https://github.com/rvandewater/YAIB for more info). We'll cover:

1. **Data Loading**: Loading dynamic measurements, static features, and outcomes from parquet files
2. **Train/Test Split**: Proper group-level splitting to prevent data leakage
3. **Multi-Step Pipeline**: 
   - Missing value imputation (forward fill + zero fill)
   - Feature scaling (standardization)
   - Historical feature engineering (rolling mean and max)
   - Custom domain-specific features
4. **Baking the Data**: Applying the preprocessing pipeline to both training and test sets
5. **Model Training**: Using the preprocessed data to train a machine learning model

The pipeline uses **Polars** for high-performance data processing, with ReciPies handling all preprocessing steps while maintaining column role information throughout the transformation pipeline.


## 1. Load ICU Data

We start by loading the ICU demo data, which consists of three components:
- **Dynamic data**: Time-varying measurements (vitals, lab values) recorded at regular intervals
- **Static data**: Patient-level features that don't change over time (age, sex, height, weight)
- **Outcome data**: The target variable we want to predict (mortality at 24 hours)

Let's examine the structure of each dataset.


In [6]:
import numpy as np
import polars as pl
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

from recipies import Ingredients, Recipe
from recipies.selector import all_predictors, all_numeric_predictors, has_role, has_type, all_of
from recipies.step import StepImputeFill, StepHistorical, StepScale, StepFunction, Accumulator, StepSklearn

dynamic_data = pl.read_parquet("../../examples/icu_demo_data/mortality24/eicu_demo/dyn.parquet")
static_data = pl.read_parquet("../../examples/icu_demo_data/mortality24/eicu_demo/sta.parquet")
outcome = pl.read_parquet("../../examples/icu_demo_data/mortality24/eicu_demo/outc.parquet")
print("Columns:")
print(f"dynamic: {dynamic_data.columns}")
print(f"static: {static_data.columns}")
print(f"outcome: {outcome.columns}")
print("Shapes:")
print(f"dynamic: {dynamic_data.shape}")
print(f"static: {static_data.shape}")
print(f"outcome: {outcome.shape}")
print("Heads:")
display(dynamic_data.head())
display(static_data.head())
display(outcome.head())

Columns:
dynamic: ['stay_id', 'time', 'alb', 'alp', 'alt', 'ast', 'be', 'bicar', 'bili', 'bili_dir', 'bnd', 'bun', 'ca', 'cai', 'ck', 'ckmb', 'cl', 'crea', 'crp', 'dbp', 'fgn', 'fio2', 'glu', 'hgb', 'hr', 'inr_pt', 'k', 'lact', 'lymph', 'map', 'mch', 'mchc', 'mcv', 'methb', 'mg', 'na', 'neut', 'o2sat', 'pco2', 'ph', 'phos', 'plt', 'po2', 'ptt', 'resp', 'sbp', 'temp', 'tnt', 'urine', 'wbc']
static: ['stay_id', 'age', 'sex', 'height', 'weight']
outcome: ['stay_id', 'label']
Shapes:
dynamic: (34175, 50)
static: (1367, 5)
outcome: (1367, 2)
Heads:


stay_id,time,alb,alp,alt,ast,be,bicar,bili,bili_dir,bnd,bun,ca,cai,ck,ckmb,cl,crea,crp,dbp,fgn,fio2,glu,hgb,hr,inr_pt,k,lact,lymph,map,mch,mchc,mcv,methb,mg,na,neut,o2sat,pco2,ph,phos,plt,po2,ptt,resp,sbp,temp,tnt,urine,wbc
i32,duration[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
141765,0ms,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,87.0,null,null,null,null,83.0,null,null,null,null,108.0,null,null,null,null,null,null,null,96.0,null,null,null,null,null,null,18.0,142.0,36.711111,null,null,null
141765,1h,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,70.0,null,null,null,null,80.0,null,null,null,null,99.0,null,null,null,null,null,null,null,96.0,null,null,null,null,null,null,23.5,144.0,36.894444,null,null,null
141765,2h,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,67.0,null,null,null,null,77.0,null,null,null,null,97.0,null,null,null,null,null,null,null,96.0,null,null,null,null,null,null,23.0,139.0,null,null,null,null
141765,3h,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,67.0,null,null,null,null,79.0,null,null,null,null,99.0,null,null,null,null,null,null,null,96.0,null,null,null,null,null,null,24.0,133.0,null,null,null,null
141765,4h,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,67.0,null,null,null,null,76.0,null,null,null,null,89.0,null,null,null,null,null,null,null,96.0,null,null,null,null,null,null,25.0,120.0,null,null,null,null


stay_id,age,sex,height,weight
i32,f64,str,f64,f64
141765,87.0,"""Female""",157.5,46.5
147784,60.0,"""Female""",154.9,95.6
151179,59.0,"""Female""",149.9,null
151867,44.0,"""Male""",172.7,null
151900,66.0,"""Female""",165.1,86.8


stay_id,label
i32,i32
141765,0
147784,0
151179,0
151867,0
151900,0


## 2. Train/Test Split

**Critical**: We perform a **group-level split** at the `stay_id` level. This ensures that all records for a given patient stay are assigned to either the training or test set, preventing data leakage where information from the test set could leak into the training process.

We use an 80/20 split stratified by `stay_id`:


In [7]:
# Train/test split at the stay_id level (group-level split)
# This ensures all records for a given stay go to either train or test

# Get unique stay_ids
unique_stays = outcome.select("stay_id").unique().sample(fraction=1.0, seed=42)
n_train = int(len(unique_stays) * 0.8)
train_stay_ids = unique_stays.head(n_train)["stay_id"].to_list()
test_stay_ids = unique_stays.tail(len(unique_stays) - n_train)["stay_id"].to_list()

# Split dynamic, static, and outcome data
dynamic_train = dynamic_data.filter(pl.col("stay_id").is_in(train_stay_ids))
dynamic_test = dynamic_data.filter(pl.col("stay_id").is_in(test_stay_ids))

static_train = static_data.filter(pl.col("stay_id").is_in(train_stay_ids))
static_test = static_data.filter(pl.col("stay_id").is_in(test_stay_ids))

outcome_train = outcome.filter(pl.col("stay_id").is_in(train_stay_ids))
outcome_test = outcome.filter(pl.col("stay_id").is_in(test_stay_ids))

# Join train data
df_train = dynamic_train.join(static_train, on="stay_id", how="left")
df_train = df_train.join(outcome_train.select(["stay_id", "label"]), on="stay_id", how="left")

# Join test data
df_test = dynamic_test.join(static_test, on="stay_id", how="left")
df_test = df_test.join(outcome_test.select(["stay_id", "label"]), on="stay_id", how="left")

print(f"Train: {len(df_train)} rows, {len(train_stay_ids)} stays")
print(f"Test: {len(df_test)} rows, {len(test_stay_ids)} stays")

Train: 27325 rows, 1093 stays
Test: 6850 rows, 274 stays


In [8]:
# Quick check: verify we have the expected columns after joining
print(f"Train dataframe columns: {len(df_train.columns)}")
print(f"Test dataframe columns: {len(df_test.columns)}")

Train dataframe columns: 55
Test dataframe columns: 55


## 3. Build Preprocessing Pipeline

Now we'll create a comprehensive preprocessing pipeline using ReciPies. The pipeline includes:

1. **Role Assignment**: Define which columns are outcomes, predictors, groups (`stay_id`), and sequences (`time`)
2. **Imputation**: Forward fill followed by zero fill for any remaining missing values
3. **Feature Scaling**: Standardize numeric predictors (mean=0, std=1)
4. **Historical Features**: Create rolling mean and max aggregations over time within each stay
5. **Custom Features**: Add domain-specific features like heart rate to temperature ratio

The key advantage of ReciPies is that all transformations maintain column role information, ensuring proper handling of grouped time-series data.


In [9]:
# Initialize Ingredients
ing = Ingredients(df_train)

# Define and build the recipe
rec = Recipe(
    ing,
    outcomes=["label"],
    predictors=[c for c in ing.columns if c not in {"label", "stay_id", "time"}],
    groups=["stay_id"],
    sequences=["time"],
)

# Impute missing values forward (pre-resample)
rec.add_step(StepImputeFill(sel=all_predictors(), strategy="forward"))
rec.add_step(StepImputeFill(sel=all_predictors(), strategy="zero"))

# Scale numeric predictors at the end (after imputation)
rec.add_step(StepScale(sel=all_numeric_predictors(), with_mean=True, with_std=True))


#  Add a custom domain feature (example: hr/temp ratio) via StepFunction
def add_custom_features(ingr: Ingredients, columns) -> Ingredients:
    df_ = ingr.get_df()
    if all(col in df_.columns for col in ["hr", "temp"]):
        df_ = df_.with_columns((pl.col("hr") / pl.col("temp")).alias("hr_temp_ratio"))
        ingr.set_df(df_)
        ingr.update_role("hr_temp_ratio", "predictor")
    return ingr


rec.add_step(StepFunction(sel=has_role(["predictor"]), function=add_custom_features))

# Label encode categorical features
types = ["String", "Object", "Categorical"]
rec.add_step(StepSklearn(SimpleImputer(missing_values=np.nan, strategy="most_frequent"), sel=has_type(types)))
rec.add_step(StepSklearn(LabelEncoder(), sel=has_type(types), columnwise=True))

original_predictors = all_of(
    list(all_numeric_predictors()(ing))
)  # Capture the fixed list of original numeric predictors
# Historical features
rec.add_step(StepHistorical(sel=original_predictors, fun=Accumulator.MEAN, suffix="_mean_hist"))
rec.add_step(StepHistorical(sel=original_predictors, fun=Accumulator.MIN, suffix="_min_hist"))
rec.add_step(StepHistorical(sel=original_predictors, fun=Accumulator.MAX, suffix="_max_hist"))
rec.add_step(StepHistorical(sel=original_predictors, fun=Accumulator.VAR, suffix="_var_hist"))

# Prep and bake (fit and transform) the training data
train_baked = rec.prep()
display(train_baked.head())
print(train_baked.columns)
print(len(train_baked.columns))

sel: roles: ['predictor']


stay_id,time,alb,alp,alt,ast,be,bicar,bili,bili_dir,bnd,bun,ca,cai,ck,ckmb,cl,crea,crp,dbp,fgn,fio2,glu,hgb,hr,inr_pt,k,lact,lymph,map,mch,mchc,mcv,methb,mg,na,neut,…,cl_var_hist,crea_var_hist,crp_var_hist,dbp_var_hist,fgn_var_hist,fio2_var_hist,glu_var_hist,hgb_var_hist,hr_var_hist,inr_pt_var_hist,k_var_hist,lact_var_hist,lymph_var_hist,map_var_hist,mch_var_hist,mchc_var_hist,mcv_var_hist,methb_var_hist,mg_var_hist,na_var_hist,neut_var_hist,o2sat_var_hist,pco2_var_hist,ph_var_hist,phos_var_hist,plt_var_hist,po2_var_hist,ptt_var_hist,resp_var_hist,sbp_var_hist,temp_var_hist,tnt_var_hist,urine_var_hist,wbc_var_hist,age_var_hist,height_var_hist,weight_var_hist
i32,duration[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
141765,0ms,-0.702651,-0.50702,-0.190116,-0.176584,0.01566,-1.208786,-0.239416,-0.080349,-0.155777,-0.83607,-1.324759,-0.158882,-0.070476,-0.092577,-1.363235,-0.668814,-0.056942,1.275731,-0.178601,-0.785766,-1.404538,-1.237212,0.018267,-0.507242,-1.386818,-0.324189,-0.537402,1.326177,-1.133195,-1.181055,-1.177998,-0.231347,-0.78064,-1.412902,-0.70648,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
141765,1h,-0.702651,-0.50702,-0.190116,-0.176584,0.01566,-1.208786,-0.239416,-0.080349,-0.155777,-0.83607,-1.324759,-0.158882,-0.070476,-0.092577,-1.363235,-0.668814,-0.056942,0.409844,-0.178601,-0.785766,-1.404538,-1.237212,-0.104868,-0.507242,-1.386818,-0.324189,-0.537402,0.933889,-1.133195,-1.181055,-1.177998,-0.231347,-0.78064,-1.412902,-0.70648,…,0.0,0.0,0.0,0.37488,0.0,0.0,0.0,0.0,0.007581,0.0,0.0,0.0,0.0,0.076945,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.210681,0.001798,0.000168,0.0,0.0,0.0,0.0,0.0,0.0
141765,2h,-0.702651,-0.50702,-0.190116,-0.176584,0.01566,-1.208786,-0.239416,-0.080349,-0.155777,-0.83607,-1.324759,-0.158882,-0.070476,-0.092577,-1.363235,-0.668814,-0.056942,0.257041,-0.178601,-0.785766,-1.404538,-1.237212,-0.228004,-0.507242,-1.386818,-0.324189,-0.537402,0.846714,-1.133195,-1.181055,-1.177998,-0.231347,-0.78064,-1.412902,-0.70648,…,0.0,0.0,0.0,0.301806,0.0,0.0,0.0,0.0,0.015162,0.0,0.0,0.0,0.0,0.065229,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.128846,0.005693,0.000112,0.0,0.0,0.0,0.0,0.0,0.0
141765,3h,-0.702651,-0.50702,-0.190116,-0.176584,0.01566,-1.208786,-0.239416,-0.080349,-0.155777,-0.83607,-1.324759,-0.158882,-0.070476,-0.092577,-1.363235,-0.668814,-0.056942,0.257041,-0.178601,-0.785766,-1.404538,-1.237212,-0.145913,-0.507242,-1.386818,-0.324189,-0.537402,0.933889,-1.133195,-1.181055,-1.177998,-0.231347,-0.78064,-1.412902,-0.70648,…,0.0,0.0,0.0,0.239326,0.0,0.0,0.0,0.0,0.010529,0.0,0.0,0.0,0.0,0.046072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.107662,0.020673,0.000084,0.0,0.0,0.0,0.0,0.0,0.0
141765,4h,-0.702651,-0.50702,-0.190116,-0.176584,0.01566,-1.208786,-0.239416,-0.080349,-0.155777,-0.83607,-1.324759,-0.158882,-0.070476,-0.092577,-1.363235,-0.668814,-0.056942,0.257041,-0.178601,-0.785766,-1.404538,-1.237212,-0.269049,-0.507242,-1.386818,-0.324189,-0.537402,0.498013,-1.133195,-1.181055,-1.177998,-0.231347,-0.78064,-1.412902,-0.70648,…,0.0,0.0,0.0,0.19665,0.0,0.0,0.0,0.0,0.012635,0.0,0.0,0.0,0.0,0.087014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.103774,0.083862,0.000067,0.0,0.0,0.0,0.0,0.0,0.0


['stay_id', 'time', 'alb', 'alp', 'alt', 'ast', 'be', 'bicar', 'bili', 'bili_dir', 'bnd', 'bun', 'ca', 'cai', 'ck', 'ckmb', 'cl', 'crea', 'crp', 'dbp', 'fgn', 'fio2', 'glu', 'hgb', 'hr', 'inr_pt', 'k', 'lact', 'lymph', 'map', 'mch', 'mchc', 'mcv', 'methb', 'mg', 'na', 'neut', 'o2sat', 'pco2', 'ph', 'phos', 'plt', 'po2', 'ptt', 'resp', 'sbp', 'temp', 'tnt', 'urine', 'wbc', 'age', 'sex', 'height', 'weight', 'label', 'hr_temp_ratio', 'alb_mean_hist', 'alp_mean_hist', 'alt_mean_hist', 'ast_mean_hist', 'be_mean_hist', 'bicar_mean_hist', 'bili_mean_hist', 'bili_dir_mean_hist', 'bnd_mean_hist', 'bun_mean_hist', 'ca_mean_hist', 'cai_mean_hist', 'ck_mean_hist', 'ckmb_mean_hist', 'cl_mean_hist', 'crea_mean_hist', 'crp_mean_hist', 'dbp_mean_hist', 'fgn_mean_hist', 'fio2_mean_hist', 'glu_mean_hist', 'hgb_mean_hist', 'hr_mean_hist', 'inr_pt_mean_hist', 'k_mean_hist', 'lact_mean_hist', 'lymph_mean_hist', 'map_mean_hist', 'mch_mean_hist', 'mchc_mean_hist', 'mcv_mean_hist', 'methb_mean_hist', 'mg_mean

## 4. Apply Pipeline to Test Data

Once the recipe is fitted on the training data using `prep()`, we can apply the same transformations to the test data using `bake()`. This ensures:

- **No data leakage**: Test data statistics are never used to fit the pipeline
- **Consistent transformations**: The same preprocessing steps are applied identically to both datasets
- **Reproducibility**: The fitted recipe can be saved and reused on new data

The `bake()` method applies all fitted transformations without refitting, ensuring the test set is processed identically to how the training set was processed.


In [10]:
test_baked = rec.bake(df_test)
display(test_baked.head())
print(test_baked.columns)
print(len(test_baked.columns))

stay_id,time,alb,alp,alt,ast,be,bicar,bili,bili_dir,bnd,bun,ca,cai,ck,ckmb,cl,crea,crp,dbp,fgn,fio2,glu,hgb,hr,inr_pt,k,lact,lymph,map,mch,mchc,mcv,methb,mg,na,neut,…,cl_var_hist,crea_var_hist,crp_var_hist,dbp_var_hist,fgn_var_hist,fio2_var_hist,glu_var_hist,hgb_var_hist,hr_var_hist,inr_pt_var_hist,k_var_hist,lact_var_hist,lymph_var_hist,map_var_hist,mch_var_hist,mchc_var_hist,mcv_var_hist,methb_var_hist,mg_var_hist,na_var_hist,neut_var_hist,o2sat_var_hist,pco2_var_hist,ph_var_hist,phos_var_hist,plt_var_hist,po2_var_hist,ptt_var_hist,resp_var_hist,sbp_var_hist,temp_var_hist,tnt_var_hist,urine_var_hist,wbc_var_hist,age_var_hist,height_var_hist,weight_var_hist
i32,duration[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
157016,0ms,-0.702651,-0.50702,-0.190116,-0.176584,0.01566,-1.208786,-0.239416,-0.080349,-0.155777,-0.83607,-1.324759,-0.158882,-0.070476,-0.092577,-1.363235,-0.668814,-0.056942,1.3776,-0.178601,-0.785766,-1.404538,-1.237212,0.264538,-0.507242,-1.386818,-0.324189,-0.537402,1.282589,-1.133195,-1.181055,-1.177998,-0.231347,-0.78064,-1.412902,-0.70648,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
157016,1h,-0.702651,-0.50702,-0.190116,-0.176584,0.01566,-1.208786,-0.239416,-0.080349,-0.155777,-0.83607,-1.324759,-0.158882,-0.070476,-0.092577,-1.363235,-0.668814,-0.056942,1.122928,-0.178601,-0.785766,-1.404538,-1.237212,0.244015,-0.507242,-1.386818,-0.324189,-0.537402,0.977477,-1.133195,-1.181055,-1.177998,-0.231347,-0.78064,-1.412902,-0.70648,…,0.0,0.0,0.0,0.032429,0.0,0.0,0.0,0.0,0.000211,0.0,0.0,0.0,0.0,0.046547,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001384,0.0,0.0,0.0,0.0,0.0,0.0,0.111435,0.018988,0.0,0.0,0.0,0.0,0.0,0.0,0.0
157016,2h,-0.702651,-0.50702,-0.190116,-0.176584,0.01566,-1.208786,-0.239416,-0.080349,-0.155777,-0.83607,-1.324759,-0.158882,-0.070476,-0.092577,-1.363235,-0.668814,-0.056942,1.173862,-0.178601,-0.785766,-1.404538,-1.237212,-0.330617,-0.507242,-1.386818,-0.324189,-0.537402,1.130033,-1.133195,-1.181055,-1.177998,-0.231347,-0.78064,-1.412902,-0.70648,…,0.0,0.0,0.0,0.01816,0.0,0.0,0.0,0.0,0.114139,0.0,0.0,0.0,0.0,0.023273,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000923,0.0,0.0,0.0,0.0,0.0,0.0,0.28323,0.009962,0.0,0.0,0.0,0.0,0.0,0.0,0.0
157016,3h,-0.702651,-0.50702,-0.190116,-0.176584,0.01566,-1.208786,-0.239416,-0.080349,-0.155777,-0.83607,-1.324759,-0.158882,-0.070476,-0.092577,-1.363235,-0.668814,-0.056942,1.530404,-0.178601,-0.785766,-1.404538,-1.237212,-0.351139,-0.507242,-1.386818,-0.324189,-0.537402,1.413352,-1.133195,-1.181055,-1.177998,-0.231347,-0.78064,-1.412902,-0.70648,…,0.0,0.0,0.0,0.035456,0.0,0.0,0.0,0.0,0.11821,0.0,0.0,0.0,0.0,0.035583,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000692,0.0,0.0,0.0,0.0,0.0,0.0,0.264657,0.018726,0.001668,0.0,0.0,0.0,0.0,0.0,0.0
157016,4h,0.83931,2.158941,-0.009931,-0.041443,0.01566,0.709533,-0.070099,-0.080349,-0.155777,-0.559254,0.842172,-0.158882,-0.070476,-0.092577,0.753987,-0.257529,-0.056942,1.530404,-0.178601,-0.785766,-0.287432,-1.237212,-0.043301,-0.507242,0.673432,-0.324189,-0.537402,1.413352,-1.133195,-1.181055,-1.177998,-0.231347,-0.78064,0.724588,-0.70648,…,0.896526,0.033831,0.0,0.037099,0.0,0.0,0.249585,0.0,0.088658,0.0,0.848926,0.0,0.0,0.035718,0.0,0.0,0.0,0.0,0.0,0.913773,0.0,0.000554,0.0,0.0,0.0,0.0,0.0,0.0,0.23262,0.019482,0.002002,0.0,0.0,0.0,0.0,0.0,0.0


['stay_id', 'time', 'alb', 'alp', 'alt', 'ast', 'be', 'bicar', 'bili', 'bili_dir', 'bnd', 'bun', 'ca', 'cai', 'ck', 'ckmb', 'cl', 'crea', 'crp', 'dbp', 'fgn', 'fio2', 'glu', 'hgb', 'hr', 'inr_pt', 'k', 'lact', 'lymph', 'map', 'mch', 'mchc', 'mcv', 'methb', 'mg', 'na', 'neut', 'o2sat', 'pco2', 'ph', 'phos', 'plt', 'po2', 'ptt', 'resp', 'sbp', 'temp', 'tnt', 'urine', 'wbc', 'age', 'sex', 'height', 'weight', 'label', 'hr_temp_ratio', 'alb_mean_hist', 'alp_mean_hist', 'alt_mean_hist', 'ast_mean_hist', 'be_mean_hist', 'bicar_mean_hist', 'bili_mean_hist', 'bili_dir_mean_hist', 'bnd_mean_hist', 'bun_mean_hist', 'ca_mean_hist', 'cai_mean_hist', 'ck_mean_hist', 'ckmb_mean_hist', 'cl_mean_hist', 'crea_mean_hist', 'crp_mean_hist', 'dbp_mean_hist', 'fgn_mean_hist', 'fio2_mean_hist', 'glu_mean_hist', 'hgb_mean_hist', 'hr_mean_hist', 'inr_pt_mean_hist', 'k_mean_hist', 'lact_mean_hist', 'lymph_mean_hist', 'map_mean_hist', 'mch_mean_hist', 'mchc_mean_hist', 'mcv_mean_hist', 'methb_mean_hist', 'mg_mean

## 5. Train a Machine Learning Model

With our preprocessed data ready, we can now train a machine learning model. The preprocessed dataframes contain:
- All original features (scaled and imputed)
- Historical aggregated features 
- One-hot encoded categorical variables
- Custom domain features (e.g., hr/temp ratio)

For demonstration, we'll use a simple logistic regression model, but you can use any scikit-learn compatible model or more advanced methods like XGBoost, LightGBM, or neural networks.


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
import numpy as np

# Extract features and labels
# Exclude outcome, group, and sequence columns from features
feature_cols = [c for c in train_baked.columns if c not in ["label", "stay_id", "time"]]

X_train = train_baked.select(feature_cols).to_numpy()
y_train = train_baked.select("label").to_numpy().ravel()

X_test = test_baked.select(feature_cols).to_numpy()
y_test = test_baked.select("label").to_numpy().ravel()

# Handle any remaining NaN values (should be minimal after preprocessing)
X_train = np.nan_to_num(X_train, nan=0.0)
X_test = np.nan_to_num(X_test, nan=0.0)

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"Class distribution (train): {np.bincount(y_train)}")
print(f"Class distribution (test): {np.bincount(y_test)}")

# Train model
model = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
model.fit(X_train, y_train)

# Predictions
y_train_pred = model.predict_proba(X_train)[:, 1]
y_test_pred = model.predict_proba(X_test)[:, 1]

# Evaluate
train_auc = roc_auc_score(y_train, y_train_pred)
test_auc = roc_auc_score(y_test, y_test_pred)

print("\nModel Performance:")
print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test, model.predict(X_test), target_names=["No Mortality", "Mortality"]))

Training set: 27325 samples, 257 features
Test set: 6850 samples, 257 features
Class distribution (train): [25875  1450]
Class distribution (test): [6550  300]

Model Performance:
Train AUC: 0.9516
Test AUC: 0.5443

Classification Report (Test Set):
              precision    recall  f1-score   support

No Mortality       0.96      0.84      0.89      6550
   Mortality       0.06      0.23      0.10       300

    accuracy                           0.81      6850
   macro avg       0.51      0.53      0.50      6850
weighted avg       0.92      0.81      0.86      6850

